# Chapter 4: AI Data Architecture Patterns

This notebook complements Chapter 4 by turning architecture trade-offs into scenario comparisons across both reference systems.

The current chapter now treats five practical patterns:
- centralized analytical core
- hybrid lakehouse platform
- domain-oriented data platform
- retrieval-centered AI platform
- prompt- and context-centered AI platform

The notebook is meant to answer `when does this pattern fit?` rather than `which pattern is most fashionable?`

In [ ]:
from pathlib import Path
import sys

import pandas as pd

def find_companion_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not find companion repository root.")


def ensure_path_exists(path: Path, kind: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected {kind}: {path}. "
            "Make sure you are running this notebook from inside the companion repository."
        )
    return path

NOTEBOOK_DIR = Path.cwd()
COMPANION_ROOT = find_companion_root(NOTEBOOK_DIR)
patterns_module_path = ensure_path_exists(
    COMPANION_ROOT / "code" / "architecture" / "patterns.py",
    "helper module",
)
sys.path.insert(0, str(COMPANION_ROOT / "code"))

from architecture.patterns import (
    build_recommendation_table,
    load_architecture_scenarios,
    recommend_architecture,
    summarize_patterns,
)

pd.set_option("display.max_columns", None)
print("Companion root located successfully.")

### Why the Setup Code Matters

The helper module encodes the same decision axes the chapter discusses: latency pressure, governance load, team structure, retrieval dependency, and feature or prompt reuse. The point is not to trust the helper blindly. The point is to make the architecture discussion explicit enough that you can argue with it.

## Part 1: Load Architecture Scenarios

The scenarios vary by latency pressure, governance load, team topology, retrieval dependency, and prompt or context complexity. Keep both recurring tracks in mind as you read them: retail ranking usually stresses feature freshness, while the support-assistant track stresses retrieval, prompt, and tool-path control.

In [ ]:
scenario_path = ensure_path_exists(
    COMPANION_ROOT / 'datasets' / 'architecture' / 'sample_architecture_scenarios.csv',
    'dataset',
)
scenarios_df = load_architecture_scenarios(scenario_path)
scenarios_df['cost_pressure'] = scenarios_df.apply(
    lambda row: 'high'
    if row['latency_profile'] == 'real_time' or (row['raw_multimodal'] == 'high' and row['team_count'] >= 9)
    else ('medium' if row['ownership_model'] != 'centralized' or row['raw_multimodal'] == 'medium' else 'low'),
    axis=1,
)
print(f"Loaded {len(scenarios_df)} architecture scenarios from {scenario_path.name}")
scenarios_df[[
    'scenario_name',
    'latency_profile',
    'ownership_model',
    'raw_multimodal',
    'governance_pressure',
    'online_serving_criticality',
    'feature_reuse_need',
    'cost_pressure',
]].sort_values('scenario_name').reset_index(drop=True)


### What to Notice in the Scenario Table

Read each scenario as a bundle of pressures rather than as an industry label. The important question is which combination of latency, governance, ownership, and retrieval or prompt complexity is forcing the architecture to change.


## Part 2: Summarize Scenario Pressure

This summary shows how latency and ownership expectations cluster across the scenarios.

In [ ]:
pattern_summary = summarize_patterns(scenarios_df)
pattern_summary

### Interpreting the Pressure Summary

The summary is useful when it reveals clusters. If many scenarios share the same latency and ownership shape, you are looking at a reusable platform pattern rather than a one-off project preference.


## Part 3: Build Architecture Recommendations

Use the recommendations like a comparison matrix, not a verdict. For each scenario, ask:

- Use when what pressure dominates?
- Avoid when what platform discipline is still missing?
- What is the main strength?
- What is the main weakness?
- What is the first likely AI failure mode?

In [ ]:
recommendation_df = build_recommendation_table(scenarios_df)
minimum_viable_map = {
    'prompt_context_centered_ai_platform': 'retrieval_centered_ai_platform',
    'retrieval_centered_ai_platform': 'hybrid_lakehouse_platform',
    'domain_oriented_data_platform': 'hybrid_lakehouse_platform',
    'hybrid_lakehouse_platform': 'centralized_analytical_core',
    'centralized_analytical_core': 'centralized_analytical_core',
}
recommendation_df['minimum_viable_architecture'] = recommendation_df['reference_pattern'].map(minimum_viable_map)
recommendation_df[[
    'scenario_name',
    'reference_pattern',
    'minimum_viable_architecture',
    'storage_pattern',
    'processing_pattern',
    'ownership_pattern',
    'feature_pattern',
    'required_layers',
]]


### Interpreting the Recommendation Output

The recommendation table should be read as an argument, not as an oracle. Each row is making a claim about where storage, processing, ownership, and feature or prompt reuse pressure are likely to land if the architecture stays disciplined.


## Part 4: Inspect a Single Scenario

A one-off recommendation is often the fastest way to pressure-test a design discussion with a team.

In [ ]:
custom_case = recommend_architecture(
    latency_profile="near_real_time",
    raw_multimodal="high",
    team_count=10,
    governance_pressure=8,
    online_serving_criticality=7,
    feature_reuse_need=8,
)
pd.Series(custom_case)

### How to Read the Single-Scenario Result

A single-scenario recommendation is most useful when you compare it to the architecture you would have chosen informally. If the output differs, ask which pressure you underweighted: serving criticality, multimodal state, governance load, or team coordination.


## Part 4A: Support-Assistant Scenario Under Prompt and Tool Risk

The helper becomes more useful when one scenario makes prompt complexity and tool-use risk explicit instead of leaving them implicit behind the generic retrieval case.

In [ ]:
support_assistant_case = pd.Series(recommend_architecture(
    latency_profile='near_real_time',
    raw_multimodal='high',
    team_count=7,
    governance_pressure=5,
    online_serving_criticality=9,
    feature_reuse_need=9,
))
support_assistant_case

### What the Support-Assistant Scenario Should Tell You

This case should push the notebook toward the prompt- and context-centered pattern because the runtime contract now depends on prompt registries, permission-aware retrieval, tool policy, trace logging, and release evidence rather than on corpus retrieval alone.

## Part 5: Compare Two Reference Cases

This contrast is most useful when one scenario looks like retail ranking and the other looks like an enterprise support assistant. The architecture should change because the contracts change: feature-serving skew dominates one system, while prompt sprawl, permission-aware retrieval, and context traceability dominate the other.

In [ ]:
comparison = recommendation_df[
    recommendation_df["scenario_name"].isin(
        ["marketing_reporting_with_ml", "retail_recommendations"]
    )
][[
    "scenario_name",
    "industry",
    "reference_pattern",
    "required_layers",
    "storage_pattern",
    "processing_pattern",
    "ownership_pattern",
    "feature_pattern",
]]
comparison

### How to Read the Recommendation Tables

Use the outputs comparatively. For each scenario, ask five questions:

- What pressure dominates?
- What pattern is being recommended?
- What would break first if the recommendation were ignored?
- What extra layer appears when retrieval or prompt/context behavior becomes operational?
- Which parts of the architecture are product requirements and which are premature platform ambition?

That keeps the notebook tied to the chapter's real goal: architecture selection as disciplined trade-off, not label collecting.

## Part 5B: What Breaks First if We Choose the Simpler Pattern?

Architecture judgment gets more practical when the notebook asks what fails first under the tempting simpler pattern, not only what the preferred pattern is.

In [ ]:
misfit_architecture_review = pd.DataFrame([
    {
        'scenario': 'retail_recommendations',
        'preferred_architecture': 'hybrid_lakehouse_platform',
        'tempting_simpler_pattern': 'centralized_analytical_core',
        'what_breaks_first': 'feature freshness and online-serving alignment drift apart under live ranking pressure',
    },
    {
        'scenario': 'support_assistant_with_tools',
        'preferred_architecture': 'prompt_context_centered_ai_platform',
        'tempting_simpler_pattern': 'retrieval_centered_ai_platform',
        'what_breaks_first': 'prompt approvals, tool permissions, and context traceability sprawl across local code',
    },
])
misfit_architecture_review

### Why the Misfit Review Matters

A simpler pattern is often the right starting point. It becomes the wrong choice when the notebook can already name the first contract it will break under real latency, retrieval, or prompt-context pressure.

## Part 5C: Inspect the Architecture Decision Artifact

The chapter names a decision matrix because architecture selection should survive beyond a one-off recommendation table. Use the artifact below as a compact review aid when you want to defend a pattern choice outside the notebook.

In [ ]:
decision_matrix_path = ensure_path_exists(
    COMPANION_ROOT / "artifacts" / "architecture" / "decision_matrix.csv",
    "artifact",
)
decision_matrix_df = pd.read_csv(decision_matrix_path)
print(decision_matrix_df)
print("\npatterns.py")
print("-" * 60)
print("\n".join(patterns_module_path.read_text(encoding="utf-8").splitlines()[:18]))

## Exercises

### Concept check
- Choose an architecture pattern for a reporting-heavy ML program, a near-real-time retail ranker, a strict-governance healthcare system, a support assistant with tools, and a multi-domain company. State the dominant pressure and the risk if the system is underbuilt or overbuilt.

### Diagnostic scenario
- Explain what breaks first when a support assistant with prompt templates, permission-aware retrieval, tool calls, and benchmark conversations is forced into a centralized warehouse-centered architecture.

### Artifact-building exercise
- Produce both an ideal architecture and a minimum viable architecture for one scenario. State what the minimum version still does not solve.

### Notebook extension
- Use the cost-pressure, minimum-viable, and misfit outputs to write a short memo on which layer you would postpone first under budget pressure and what failure you are accepting temporarily.